# 3. Feature Engineering

In [1]:
from chess_eval import *

The most natural indicator that one would consider to predict the advantage of the players is the piece count: how many pieces does each player have. So, we code the first transformer to extract the piece count from the FEN.

Right away, we test our simple model with only these basic features. We will be considering a simple Random Forest and Linear Regressor.

Since the dataset is fairly balanced (54%-37), we will consider the plain sign accuracy of our prediction. For now, we are just checking the evaluation's sign because the RMSE is not yet a good indicator for model fitness.

We have analyzed the model’s feature importance to identify which aspects of material and position—such as specific piece counts or values—have contributed the most to predicting the evaluation. It allowed us to understand which factors have most strongly driven the model’s decisions.

Pawns located near the king is the pillar for king's safety. We all know that _the more, the merrier_. Here we put in practice this well-known idiom (the more pawns you have near your king, the safer and better protected he is). This class extracts positional board-control features such as central control, open files, rook placement, piece alignment, protected advanced pawns, and total controlled squares to quantify strategic dominance.

As we add more features, the model performance stays roughly the same. We need to engineer more informative features to improve performance.

In chess, the mobility of the pieces is a crucial factor to consider, because if a piece has no mobility, it is useless to the player. We will consider the mobility of each piece in the game.

The BoardControl class is important because it encodes essential positional factors that material alone cannot capture. Control of central squares, open and semi-open files, rook placement, protected advanced pawns, piece alignment, and total controlled squares all describe how active, coordinated, and dominant each side’s pieces are. These concepts directly influence long-term strategic advantages such as space, mobility, initiative, and attack potential.

In chess, relations such as threats created, hanging pieces, and undefended pieces are important because they show tactical opportunities, immediate risks, and positional weaknesses, and by counting them and weighting them in points, they provide a clear measure of material and strategic advantage, helping players or engines evaluate the position accurately.

In this section, we will address the remaining features that are not grouped semantically. These are the player to move, the number of moves, and the number of halfmoves since the last capture or pawn advance (halfmove clock), denoted as $B$, $E$, and $F$, respectively.

## 3.x Extracting Piece's Values

In [ ]:
dm = load_dataset("one")
mm = ModelManager(LinearRegression())
mm.fit(dm)
mm.predict(dm)
metrics = MetricsManager(mm=mm)
coefs = pd.Series(mm.model.coef_, index=dm.X_train.columns)
intercept = mm.model.intercept_
piece_types = ["Pawn","Knight","Bishop","Rook","Queen"]

avg_abs = {}
for p in piece_types:
    cols = [c for c in coefs.index if p in c]
    vals = coefs[cols].abs()
    avg_abs[p] = vals.mean()

avg_abs = pd.Series(avg_abs)

pawn_val = avg_abs["Pawn"]
queen_val = avg_abs["Queen"]

scale = (900 - 100) / (queen_val - pawn_val)
shift = 100 - pawn_val * scale

normalized = avg_abs * scale + shift
normalized